## TODO
* Автоэнкодер
* Cross-domain
* Beam search with lp and cp
* SRU
* Визуализация Attn
* replace_unk по attention'у http://opennmt.net/OpenNMT/translation/unknowns/

## Tutorials
* http://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html
* https://github.com/spro/practical-pytorch/blob/master/seq2seq-translation/seq2seq-translation-batched.ipynb

## Articles
* Teaching neural networks to point to improve language modeling and translation: https://einstein.ai/research/teaching-neural-networks-to-point-to-improve-language-modeling-and-translation
* Training RNNs as Fast as CNNs : https://arxiv.org/abs/1709.02755
* Beam Search Strategies for Neural Machine Translation: https://arxiv.org/abs/1702.01806
* Unsupervised Machine Translation Using Monolingual Corpora Only: https://arxiv.org/abs/1711.00043
* Unsupervised Neural Machine Translation: https://arxiv.org/abs/1710.11041
* NIPS 2016 Tutorial: Generative Adversarial Networks: https://arxiv.org/pdf/1701.00160.pdf

## Repos
* https://github.com/facebookresearch/MUSE
* https://github.com/OpenNMT/OpenNMT-py

In [1]:
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F
from torch import optim
from collections import Counter, namedtuple
import pickle
import os
import re
import random
import time
import numpy as np
from typing import List, Tuple
from torch.nn.utils.rnn import pack_padded_sequence as pack
from torch.nn.utils.rnn import pad_packed_sequence as unpack
from collections import Counter
from gensim.models.keyedvectors import KeyedVectors
from models import EncoderRNN, AttnDecoderRNN, Generator

from utils.vocabulary import Vocabulary
from utils.tqdm import tqdm_open
from batch import Batch, OneLangBatch, BatchGenerator, OneLangBatchGenerator

use_cuda = torch.cuda.is_available()
print(use_cuda)
%load_ext autoreload
%autoreload 2

True


In [14]:
VAL_FILENAME = "src.txt"
OUTPUT_FILENAME = "pred.txt"
with open(VAL_FILENAME, "r", encoding='utf-8') as r:
    with open(OUTPUT_FILENAME, "w", encoding='utf-8') as w:
        for line in r:
            line = line.strip()
            translation = translate(model, line)
            w.write(" ".join(translation) + "\n")
!perl multi-bleu.perl ref.txt < pred.txt

NameError: name 'model' is not defined

In [20]:
# M1
class NNModel:
    def __init__(self, model):
        self.model = model
    
    def translate(self, sentence):
        return translate(self.model, sentence)

class WordByWordModel:
    def __init__(self, bilingual_dict_filename):
        self.bilingual_dict_filename = bilingual_dict_filename
        self.src_vocabulary = Vocabulary(language="en")
        self.tgt_vocabulary = Vocabulary(language="de")
        self.src2tgt = {0:0, 1:1, 2:2, 3:3}
        self.tgt2src = {0:0, 1:1, 2:2, 3:3}
        with open(self.bilingual_dict_filename, "r", encoding='utf-8') as r:
            for line in r:
                src_word, tgt_word = line.strip().split()
                self.src_vocabulary.add_word(src_word)
                self.tgt_vocabulary.add_word(tgt_word)
                src_index = self.src_vocabulary.get_index(src_word)
                tgt_index = self.tgt_vocabulary.get_index(tgt_word)
                self.src2tgt[src_index] = tgt_index
                self.tgt2src[tgt_index] = src_index
                
    def translate_src2tgt(self, indices):
        result = []
        for src_index in indices:
            if src_index in self.src2tgt:
                result.append(self.src2tgt[src_index])
            else:
                result.append(self.tgt_vocabulary.get_ukn())
        return result
    
    def translate_tgt2src(self, indices):
        result = []
        for tgt_index in indices:
            if tgt_index in self.tgt2src:
                result.append(self.tgt2src[tgt_index])
            else:
                result.append(self.src_vocabulary.get_ukn())
        return result
    
    def translate_src2tgt_sentence(self, sentence):
        indices = []
        for word in sentence.split():
            word = word.lower()
            indices.append(self.src_vocabulary.get_index(word))
        result = self.translate_src2tgt(indices)
        result = [self.tgt_vocabulary.get_word(i) for i in result]
        result.append("<EOS>")
        return result
    
    def translate_tgt2src_sentence(self, sentence):
        indices = []
        for word in sentence.split():
            word = word.lower()
            indices.append(self.tgt_vocabulary.get_index(word))
        result = self.translate_tgt2src(indices)
        result = [self.src_vocabulary.get_word(i) for i in result]
        result.append("<EOS>")
        return result

BILINGUAL_DICT = "models/en-de.txt"
M0 = WordByWordModel(BILINGUAL_DICT)
current_model = M0

In [21]:
# M1 evaluate
VAL_FILENAME = "src.txt"
OUTPUT_FILENAME = "pred.txt"
with open(VAL_FILENAME, "r", encoding='utf-8') as r:
    with open(OUTPUT_FILENAME, "w", encoding='utf-8') as w:
        for line in r:
            line = line.strip()
            translation = current_model.translate_src2tgt_sentence(line)
            w.write(" ".join(translation[:-1]) + "\n")
!perl multi-bleu.perl -lc ref.txt < pred.txt

Use of uninitialized value in division (/) at multi-bleu.perl line 139, <STDIN> line 162.
BLEU = 0.00, 11.9/1.2/0.3/0.0 (BP=1.000, ratio=1.101, hyp_len=2010, ref_len=1826)
It is in-advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.


In [22]:
class Discriminator(nn.Module):
    def __init__(self, max_length, encoder_hidden_size, hidden_size=1024, n_layers=3, activation=F.leaky_relu):
        super(Discriminator, self).__init__()
        
        self.encoder_hidden_size = encoder_hidden_size
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.activation = activation
        self.max_length = max_length
        
        layers = []
        layers.append(nn.Linear(encoder_hidden_size*max_length, hidden_size))
        for i in range(n_layers-1):
            layers.append(nn.Linear(hidden_size, hidden_size))
        self.layers = nn.ModuleList(layers)
        self.out = nn.Linear(hidden_size, 2)

    def forward(self, encoder_output):
        max_length = encoder_output.size(0)
        batch_size = encoder_output.size(1)
        output = encoder_output.transpose(0, 1).contiguous().view(batch_size, max_length*self.encoder_hidden_size) 
        output = F.pad(output, (0, (self.max_length-max_length)*self.encoder_hidden_size), "constant", 0)
        # S = batch_size, max_length * encoder_hidden_size
        for i in range(self.n_layers):
            output = self.layers[i](output)
            output = self.activation(output)
        return F.log_softmax(self.out(output), dim=1)

In [23]:
class UNMT(nn.Module):
    def __init__(self, embedding_dim, src_vocabulary, tgt_vocabulary, hidden_size,
                 encoder_n_layers=3, decoder_n_layers=3, dropout=0.1, max_length=50):
        super(UNMT, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.src_size = src_vocabulary.size()
        self.tgt_size = tgt_vocabulary.size()
        self.hidden_size = hidden_size
        self.encoder_n_layers = encoder_n_layers
        self.decoder_n_layers = decoder_n_layers
        self.dropout = dropout
        self.max_length = max_length
        self.src_vocabulary = src_vocabulary
        self.tgt_vocabulary = tgt_vocabulary
        
        self.src_encoder = EncoderRNN(self.src_size, embedding_dim, hidden_size, dropout=dropout, 
                                      n_layers=encoder_n_layers)
        self.tgt_encoder = EncoderRNN(self.tgt_size, embedding_dim, hidden_size, dropout=dropout, 
                                      n_layers=encoder_n_layers)
        self.src_decoder = AttnDecoderRNN(embedding_dim, hidden_size, self.src_size, dropout=dropout, 
                                          max_length=max_length, n_layers=decoder_n_layers)
        self.tgt_decoder = AttnDecoderRNN(embedding_dim, hidden_size, self.tgt_size, dropout=dropout, 
                                          max_length=max_length, n_layers=decoder_n_layers)
        self.src_generator = Generator(hidden_size, self.src_size)
        self.tgt_generator = Generator(hidden_size, self.tgt_size)
        self.discriminator = Discriminator(self.max_length, self.hidden_size)
        
    def load_embeddings(self, src_embeddings, tgt_embeddings, enable_training=False):
        aligned_src_embeddings = torch.randn(self.src_vocabulary.size(), 300)
        for i, word in enumerate(self.src_vocabulary.index2word):
            if word in src_embeddings.wv and i > 3:
                aligned_src_embeddings[i] = torch.FloatTensor(src_embeddings.wv[word])
                
        aligned_tgt_embeddings = torch.randn(self.tgt_vocabulary.size(), 300)
        for i, word in enumerate(self.tgt_vocabulary.index2word):
            if word in tgt_embeddings.wv and i > 3:
                aligned_tgt_embeddings[i] = torch.FloatTensor(tgt_embeddings.wv[word])
                
        self.src_encoder.embedding.weight = nn.Parameter(aligned_src_embeddings)
        self.tgt_encoder.embedding.weight = nn.Parameter(aligned_tgt_embeddings)
        self.src_decoder.embedding.weight = nn.Parameter(aligned_src_embeddings)
        self.tgt_decoder.embedding.weight = nn.Parameter(aligned_tgt_embeddings)
        
        if not enable_training:
            self.src_encoder.embedding.weight.requires_grad = False
            self.tgt_encoder.embedding.weight.requires_grad = False
            self.src_decoder.embedding.weight.requires_grad = False
            self.tgt_decoder.embedding.weight.requires_grad = False
    
    def forward(self, batch: Batch, noisy_batch: Batch, translated_noisy_batch: Batch, 
                batch_size, criterion, src_vocabulary, tgt_vocabulary):
        if use_cuda:
            batch = batch.cuda()
            noisy_batch = noisy_batch.cuda()
            translated_noisy_batch = translated_noisy_batch.cuda()
        
        src_adv_loss, src_auto_loss = \
            self.auto_encoder_decoder_run(self.src_encoder, self.src_decoder, self.src_generator, criterion, 
                                          noisy_batch.src_variable, noisy_batch.src_lengths, batch_size, lang="src")
            
        tgt_adv_loss, tgt_auto_loss = \
            self.auto_encoder_decoder_run(self.tgt_encoder, self.tgt_decoder, self.tgt_generator, criterion, 
                                          noisy_batch.tgt_variable, noisy_batch.tgt_lengths, batch_size, lang="tgt")
            
        cd_tgt_adv_loss, cd_tgt_cd_loss = \
            self.cd_encoder_decoder_run(self.src_encoder, self.tgt_decoder, self.tgt_generator, criterion, 
                                        translated_noisy_batch.tgt_variable, translated_noisy_batch.tgt_lengths, 
                                        batch.tgt_variable, batch_size, lang="tgt")
        
        cd_src_adv_loss, cd_src_cd_loss = \
            self.cd_encoder_decoder_run(self.tgt_encoder, self.src_decoder, self.src_generator, criterion, 
                                        translated_noisy_batch.src_variable, translated_noisy_batch.src_lengths, 
                                        batch.src_variable, batch_size, lang="src")
        
        return sum([src_adv_loss, src_auto_loss, tgt_adv_loss, tgt_auto_loss, 
                    cd_tgt_adv_loss, cd_tgt_cd_loss, cd_src_adv_loss, cd_src_cd_loss])
    
    def translate_src2tgt(self, variable, lengths, batch_size):
        return self.translate(variable, lengths, self.src_encoder, self.tgt_decoder, self.tgt_generator, batch_size)
    
    def translate_tgt2src(self, variable, lengths, batch_size):
        return self.translate(variable, lengths, self.tgt_encoder, self.src_decoder, self.src_generator, batch_size)
        
    def translate(self, variable, lengths, encoder, decoder, generator, batch_size):
        output_variable = Variable(torch.zeros(self.max_length, self.batch_size)).type(torch.LongTensor)
        output_variable = output_variable.cuda() if use_cuda else output_variable
        
        encoder_output, encoder_hidden = encoder(variable, lengths, None)
        initial_input, initial_context = decoder.init_state(batch_size)
        for t in range(max_length):
            decoder_output, decoder_hidden, attn_weights = \
                decoder(output_variable, [t+1 for i in range(batch_size)],
                        encoder_hidden, encoder_output, initial_input, initial_context)
            
            scores = generator(decoder_output[t])
            for i in range(batch_size):
                topv, topi = scores.data[i].topk(1)
                ni = topi[0]
                output_variable[t, i] = topi
        for i in range(batch_size):
            eos_index = 0
            for t in range(max_length):
                if output_variable[t, i] == 2:
                    eos_index = t
            for t in range(eos_index+1, max_length):
                output_variable[t, i] = 0
        return output_variable
        
    def auto_encoder_decoder_run(self, encoder, decoder, generator, criterion, variable, 
                                 lengths, batch_size, lang="src"):
        
        encoder_output, encoder_hidden = encoder(variable, lengths, None)
        
        # Adversarial part
        adv_criterion = nn.NLLLoss()
        log_proba = self.discriminator(encoder_output)
        if lang == "src":
            target_variable = Variable(torch.LongTensor([0, 1]))
        else:
            target_variable = Variable(torch.LongTensor([1, 0]))
        adv_loss = adv_criterion(log_proba, target_variable)
        
        # Auto part
        initial_input, initial_context = decoder.init_state(batch_size)
        decoder_output, _, _ = decoder(variable, lengths, encoder_hidden, encoder_output, 
                                       initial_input, initial_context)
        auto_loss = 0
        max_length = max(lengths)
        for t in range(max_length):
            scores = generator(decoder_output[t])
            auto_loss += criterion(scores, variable[t])
            
        return adv_loss, auto_loss

    def cd_encoder_decoder_run(self, encoder, decoder, generator, criterion, variable, lengths, 
                               gt_variable, batch_size, lang="src"):
        encoder_output, encoder_hidden = encoder(variable, lengths, None)
        
        # Adversarial part
        adv_criterion = nn.NLLLoss()
        log_proba = self.discriminator(encoder_output)
        if lang == "src":
            target_variable = Variable(torch.LongTensor([1, 0]))
        else:
            target_variable = Variable(torch.LongTensor([0, 1]))
        adv_loss = adv_criterion(log_proba, target_variable)
        
        # Cross-domain part
        initial_input, initial_context = decoder.init_state(batch_size)
        decoder_output, _, _ = decoder(variable, lengths, encoder_hidden, encoder_output, 
                                       initial_input, initial_context)
        
        cd_loss = 0
        max_length = max(lengths)
        for t in range(max_length):
            scores = generator(decoder_output[t])
            cd_loss += criterion(decoder_output, gt_variable[t])
        
        return adv_loss, cd_loss

In [32]:
class GlobalState:
    def __init__(self, bilingual_dict="models/en-de.txt", src_embeddings="models/wiki.multi.en.vec", 
                 tgt_embeddings="models/wiki.multi.de.vec", batch_size=64):
        self.current_model = WordByWordModel(bilingual_dict)
        self.src_vocabulary = self.current_model.src_vocabulary
        self.tgt_vocabulary = self.current_model.tgt_vocabulary
        self.src_word_vectors = KeyedVectors.load_word2vec_format(src_embeddings, binary=False)
        self.tgt_word_vectors = KeyedVectors.load_word2vec_format(tgt_embeddings, binary=False)
        
        self.batch_size = batch_size
        self.max_length = 50
        
        self.discriminator_optimizer = None
        self.main_optimizer = None
        
        weight = torch.ones(self.tgt_vocabulary.size())
        weight[self.tgt_vocabulary.get_pad()] = 0
        weight = weight.cuda() if use_cuda else weight
        self.tgt_criterion = nn.NLLLoss(weight, size_average=False)
        
        weight = torch.ones(self.src_vocabulary.size())
        weight[self.src_vocabulary.get_pad()] = 0
        weight = weight.cuda() if use_cuda else weight
        self.src_criterion = nn.NLLLoss(weight, size_average=False)

    def train(self, src_filenames, tgt_filenames, train_pair_filenames: List[Tuple[str, str]], val_pair_filenames: List[Tuple[str, str]],
              big_epochs: int, print_every=3000, save_every=3000):
        model = UNMT(300, self.src_vocabulary, self.tgt_vocabulary, 500)
        model.load_embeddings(self.src_word_vectors, self.tgt_word_vectors, enable_training=False)
        model = model.cuda() if use_cuda else model
        
        self.discriminator_optimizer = optim.SGD(model.discriminator.parameters(), lr=1)
        self.main_optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=1)
        
        src_batches = self.get_one_lang_batches(src_filenames)
        tgt_batches = self.get_one_lang_batches(tgt_filenames)
        train_batches = self.get_parallel_batches(train_pair_filenames)
        val_batches = self.get_parallel_batches(val_pair_filenames)
        
        print(src_batches[[0]])
        print(train_batches[0])
        print(val_batches[0])
        
        val_perm = np.random.permutation(len(val_batches))[:1000]
        val_batches = [val_batches[index] for index in val_perm]

        print(model)
        
        model_parameters = filter(lambda p: p.requires_grad, model.parameters())
        params = sum([np.prod(p.size()) for p in model_parameters])
        print("Params: ", params)
        
        print("Input:", train_batches[0].src_variable)
        print("Output:", train_batches[0].tgt_variable)

        for big_epoch in range(big_epochs):
            timer = time.time()
            print_loss_total = 0
            count_tokens = 0
            perm = np.random.permutation(count_batches)
            for epoch, batch_index in enumerate(perm):
                batch = train_batches[batch_index]
                discrimintor_loss, main_loss = self.train_batch(model, batch)
                self.current_model = model
                print(discrimintor_loss, main_loss)

                print_loss_total += main_loss
                count_tokens += sum(batch.input_lengths)
                if epoch % save_every == 0 and epoch != 0:
                    save(model, optimizer, "model.pt")
                if epoch % print_every == 0 and epoch != 0:
                    val_loss = 0
                    for val_batch in val_batches:
                        val_loss += validate_batch(model, criterion, val_batch)
                    val_loss /= len(val_batches)
                    print_loss_avg = print_loss_total / print_every
                    print_loss_total = 0
                    diff = time.time() - timer
                    timer = time.time()
                    src_speed = count_tokens / diff
                    print('%s big epoch, %s/%s, %s src tok/s, %s sec, %.4f loss, %.4f val loss' % 
                          (big_epoch, epoch, count_batches, src_speed, diff, print_loss_avg, val_loss))
                    count_tokens = 0
                    
    def get_one_lang_batches(self, filenames, lang="src"):
        vocabulary = self.src_vocabulary if lang == "src" else self.tgt_vocabulary
        batch_generator = OneLangBatchGenerator(filenames, self.batch_size, self.max_length, vocabulary)
        batches = []
        for batch in batch_generator:
            batches.append(batch)
        return batches
    
    def get_parallel_batches(self, pair_filenames):
        batch_generator = BatchGenerator(pair_filenames, self.batch_size, self.max_length, 
                                         self.src_vocabulary, self.tgt_vocabulary, use_cuda)
        batches = []
        for batch in batch_generator:
            batches.append(batch)
        return batches
        
    def train_batch(self, model, batch: Batch):
        noisy_batch = self.prepare_noisy_input(batch)
        translated_noisy_batch = self.prepare_translated_noisy_input(batch)
        
        # Disciminator step
        batch = batch.cuda()
        self.discriminator_optimizer.zero_grad()
        adv_criterion = nn.NLLLoss()
        
        src_encoder_output, _ = model.src_encoder(batch.src_variable, batch.src_lengths, None)
        log_proba = model.discriminator(src_encoder_output)
        src_variable = Variable(torch.LongTensor([0 for _ in range(self.batch_size)]))
        src_variable = src_variable.cuda() if use_cuda else src_variable
        src_adv_loss = adv_criterion(log_proba, src_variable)
        
        tgt_encoder_output, _ = model.tgt_encoder(batch.tgt_variable, batch.tgt_lengths, None)
        log_proba = model.discriminator(tgt_encoder_output)
        tgt_variable = Variable(torch.LongTensor([1 for _ in range(self.batch_size)]))
        tgt_variable = tgt_variable.cuda() if use_cuda else tgt_variable
        tgt_adv_loss = adv_criterion(log_proba,  tgt_variable)
        
        discriminator_loss = src_adv_loss + tgt_adv_loss
        discriminator_loss.backward()
        nn.utils.clip_grad_norm(model.discriminator.parameters(), 5)
        self.discriminator_optimizer.step()
        
        # Main step
        noisy_batch = noisy_batch.cuda()
        translated_noisy_batch = translated_noisy_batch.cuda()
        self.main_optimizer.zero_grad()
        loss = model(batch, noisy_batch, translated_noisy_batch, self.batch_size, 
                     criterion, self.src_vocabulary, self.tgt_vocabulary)
        loss.backward()
        nn.utils.clip_grad_norm(model.parameters(), 5)
        self.main_optimizer.step()
        
    def get_variable(self, sentence, vocabulary):
        indices = indices_from_sentence(sentence, self.src_vocabulary)
        variable = Variable(torch.zeros(self.batch_size, len(indices))).type(torch.LongTensor)
        indices = Variable(torch.LongTensor(indices))
        variable[0] = indices
        for i in range(1, batch_size):
            variable[i, 0] = self.src_vocabulary.get_eos()
        variable = variable.transpose(0, 1)
        variable = variable.cuda() if use_cuda else variable
        lengths = [len(indices)]
        lengths += [1 for _ in range(self.batch_size-1)]
        return variable, lengths
        
    def prepare_noisy_input(self, batch: Batch):
        new_src_variable, new_src_lengths = self.prepare_noisy_variable(batch.src_variable)
        new_tgt_variable, new_tgt_lengths = self.prepare_noisy_variable(batch.tgt_variable)
        return Batch(new_src_variable, new_tgt_variable, new_src_lengths, new_tgt_lengths)
    
    def prepare_translated_noisy_input(self, batch: Batch):
        new_src_variable, _ = self.prepare_translated_variable(batch.src_variable, lang="src")
        new_src_variable, new_src_lengths = self.prepare_noisy_variable(new_src_variable)
        
        new_tgt_variable, _ = self.prepare_translated_variable(batch.tgt_variable, lang="tgt")
        new_tgt_variable, new_tgt_lengths = self.prepare_noisy_variable(new_tgt_variable)
        return Batch(new_src_variable, new_tgt_variable, new_src_lengths, new_tgt_lengths)
            
    def prepare_noisy_variable(self, variable):
        assert variable.size(1) == self.batch_size
        max_length = variable.size(0)
        variable = variable.transpose(0, 1)
        new_lengths = []
        new_varibale = Variable(torch.zeros(self.batch_size, max_length)).type(torch.LongTensor)
        for b in range(self.batch_size):
            indices = [elem for elem in variable[b].data if elem != 0][:-1]
            noisy = add_noise(indices) + [2, ]
            new_lengths.append(len(noisy))
            noisy = noisy + [0 for _ in range(max_length - len(noisy))]
            new_varibale[b] = torch.LongTensor(noisy)
        return new_varibale.transpose(0, 1), new_lengths
    
    def prepare_translated_variable(self, variable, lang="src"):
        variable = variable.transpose(0, 1)
        new_sentences = []
        for b in range(self.batch_size):
            if lang == "src":
                translated = self.current_model.translate_src2tgt(list(variable[b].data))
            else:
                translated = self.current_model.translate_tgt2src(list(variable[b].data))
            new_sentences.append(translated)
        lengths = [len(sentence) for sentence in new_sentences]
        max_length = max(lengths)
        new_variable = Variable(torch.zeros(self.batch_size, max_length)).type(torch.LongTensor)
        for b in range(self.batch_size):
            current_sentence = new_sentences[b] 
            current_sentence = current_sentence + [0 for _ in range(max_length-len(current_sentence))]
            new_variable[b] = torch.LongTensor(current_sentence)
        return new_variable.transpose(0, 1), lengths
            
    @staticmethod
    def add_noise(sequence, drop_probability=0.1, shuffle_max_distance=3):
        new_sequence = [elem for elem in sequence if np.random.random_sample() > drop_probability]
        new_sequence = [x for i, x in sorted(enumerate(new_sequence), key = lambda x: x[0] + (shuffle_max_distance+1)*np.random.random())]
        return new_sequence

In [ ]:
# VAL_INPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/datasets/val-sorted.en"
# VAL_OUTPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/datasets/val-sorted.de"
# val_batch_generator = BatchGenerator([(VAL_INPUT_FILENAME, VAL_OUTPUT_FILENAME), ], 64, 50,
#                                          M0.src_vocabulary, M0.tgt_vocabulary, use_cuda)
# val_batches = []
# for batch in val_batch_generator:
#     val_batches.append(batch)
# val_perm = np.random.permutation(len(val_batches))[:1000]
# val_batches = [batch for i, batch in enumerate(val_batches) if i in val_perm]   
# batch = val_batches[900]

In [ ]:
state = GlobalState()

In [ ]:
TRAIN_INPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/datasets/train-sorted.en"
TRAIN_OUTPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/datasets/train-sorted.de"
VAL_INPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/datasets/val-sorted.en"
VAL_OUTPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/datasets/val-sorted.de"
train_pairs = [(TRAIN_INPUT_FILENAME, TRAIN_OUTPUT_FILENAME)]
val_pairs = [(VAL_INPUT_FILENAME, VAL_OUTPUT_FILENAME)]
state.train([TRAIN_INPUT_FILENAME, ], [TRAIN_OUTPUT_FILENAME, ], train_pairs, val_pairs, 3)